In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn.datasets
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

import optuna
from optuna.samplers import TPESampler
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import make_scorer, mean_squared_error

In [2]:
load_california = sklearn.datasets.fetch_california_housing()

X = pd.DataFrame(data = load_california["data"], columns = load_california["feature_names"])
Y = pd.DataFrame(data = load_california["target"], columns = load_california["target_names"])

df = pd.concat([X, Y], axis=1)

Y = Y.values.ravel()


X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.3, random_state=42,)
#  stratify  = ?



# trenujemy model z domyslnymi hiperparametrami
def train_model_KNR(X_train, X_test, Y_train, Y_test, model):

   
    md = model.fit(X_train, Y_train)
    res = md.predict(X_test)
    MSE = np.mean((res-Y_test)**2)
    r2 = r2_score(Y_test, res)
    
    print("MSE:", MSE)
    print("R2:", r2)


train_model_KNR(X_train, X_test, Y_train, Y_test, KNeighborsRegressor())

MSE: 1.136942049088978
R2: 0.1337849088797427


In [3]:
def train_modelSVR(X_train, X_test, Y_train, Y_test):
    model = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR(kernel='rbf', C=10, epsilon=0.1, gamma='scale'))
    ])
    
    model.fit(X_train, Y_train)
    pred = model.predict(X_test)
    
    mse = np.mean((Y_test - pred) ** 2)
    r2 = r2_score(Y_test, pred)
    
    print("MSE:", mse)
    print("R2:", r2)


train_modelSVR(X_train, X_test, Y_train, Y_test)

MSE: 0.3183310833686117
R2: 0.7574694430490078


In [ ]:
def train_modelRandomF(X_train, X_test, Y_train, Y_test):

    model = Pipeline([
    ("rf", RandomForestRegressor()) ])
    

    model.fit(X_train, Y_train)
    pred = model.predict(X_test)
    
    mse = np.mean((Y_test - pred) ** 2)
    r2 = r2_score(Y_test, pred)
    
    print("MSE:", mse)
    print("R2:", r2)

train_modelRandomF(X_train, X_test, Y_train, Y_test)

In [ ]:
def train_modelXGBR(X_train, X_test, Y_train, Y_test):
    
    model = XGBRegressor().fit(X_train, Y_train)
    pred = model.predict(X_test)
    
    mse = np.mean((Y_test - pred) ** 2)
    r2 = r2_score(Y_test, pred)
    
    print("MSE:", mse)
    print("R2:", r2)
    
train_modelXGBR(X_train, X_test, Y_train, Y_test)

In [ ]:
#3 Szukamy optymalnych hiperparametrow za pomoca Optuny i CV
def mse_scorer(Y_test, y_pred):
    return mean_squared_error(Y_test, y_pred)

scorer = make_scorer(mse_scorer, greater_is_better=False)

# ----------------------------
# Obiektyw Optuny (najważniejsze!)
# ----------------------------
def objective(trial):

    # Proponowane przez Optunę hiperparametry
    C = trial.suggest_float("C", 1e-3, 1e3, log=True)
    epsilon = trial.suggest_float("epsilon", 1e-4, 1.0, log=True)
    gamma = trial.suggest_float("gamma", 1e-4, 1.0, log=True)

    # Pipeline (skalowanie + SVR)
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("svr", SVR(C=C, epsilon=epsilon, gamma=gamma, kernel="rbf"))
    ])

    # Cross-validation (5-fold)
    cv = KFold(n_splits=5, shuffle=True, random_state=42)

    # Negatywne MSE (bo Optuna minimalizuje → negacja zamienia się w maksymalizację)
    scores = cross_val_score(model, X, Y, cv=cv, scoring=scorer)

    # Optuna minimalizuje → zwracamy *ujemne* MSE
    return -scores.mean()

# ----------------------------
# Uruchomienie optymalizacji
# ----------------------------
study = optuna.create_study(
    direction="minimize",   # minimalizujemy MSE
    sampler=TPESampler(seed=42)
)

study.optimize(objective, n_trials=50, n_jobs=-1)

print("Najlepsze parametry:")
print(study.best_params)